# 07 — Modelagem bônus

Comparativo adicional de algoritmos particionais no espaço **one-hot + StandardScaler**
da base IBM HR (PeopleCluster).

> Não substitui o modelo principal (K-Medoids / Gower do notebook 04).
> Serve para stress-test e sensibilidade a alternativas do scikit-learn.

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import BisectingKMeans, KMeans, MiniBatchKMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config

config.garantir_diretorios()
SEMENTE = config.SEMENTE

In [ ]:
X = np.load(config.DATA_PROCESSED / "matriz_kmeans_scaled.npy")
features = pd.read_csv(config.DATA_PROCESSED / "hr_features_cluster.csv")
avaliacao = pd.read_csv(config.DATA_PROCESSED / "hr_avaliacao.csv")
principal = pd.read_csv(config.DATA_PROCESSED / "rotulos_clusters.csv")

print("Matriz padronizada:", X.shape)

## Grade de modelos (k ∈ 2…5)

In [ ]:
modelos = [
    (
        "k-means",
        KMeans,
        {"n_clusters": [2, 3, 4, 5], "init": ["k-means++", "random"], "n_init": [10]},
    ),
    (
        "mini-batch-k-means",
        MiniBatchKMeans,
        {
            "n_clusters": [2, 3, 4, 5],
            "batch_size": [64, 128, 256],
            "n_init": [10],
        },
    ),
    (
        "bisecting-k-means",
        BisectingKMeans,
        {
            "n_clusters": [2, 3, 4, 5],
            "bisecting_strategy": ["biggest_inertia", "largest_cluster"],
        },
    ),
]

linhas = []
for nome, cls, grade in modelos:
    for params in ParameterGrid(grade):
        modelo = cls(random_state=SEMENTE, **params)
        rotulos = modelo.fit_predict(X)
        n_grupos = len(np.unique(rotulos))
        if n_grupos < 2:
            continue
        linhas.append(
            {
                "modelo": nome,
                "params": json.dumps(params, sort_keys=True),
                "n_clusters": int(params.get("n_clusters", n_grupos)),
                "silhueta": float(silhouette_score(X, rotulos)),
                "davies_bouldin": float(davies_bouldin_score(X, rotulos)),
                "menor_grupo": int(np.bincount(rotulos).min()),
                "maior_grupo": int(np.bincount(rotulos).max()),
            }
        )

resultados = pd.DataFrame(linhas).sort_values(
    ["silhueta", "davies_bouldin"], ascending=[False, True]
)
resultados.to_csv(config.TABLES / "bonus_varredura_modelos.csv", index=False)
resultados.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
resumo = resultados.groupby(["modelo", "n_clusters"])["silhueta"].max().reset_index()
sns.lineplot(data=resumo, x="n_clusters", y="silhueta", hue="modelo", marker="o", ax=ax)
ax.set_title("Melhor silhueta por modelo e k (espaço padronizado)")
plt.tight_layout()
fig.savefig(config.FIGURES / "14_bonus_silhueta.png", dpi=120, bbox_inches="tight")
fig.savefig(config.DOCS_FIGURES / "14_bonus_silhueta.png", dpi=120, bbox_inches="tight")
plt.show()

## Modelo vencedor do bônus e perfil

In [ ]:
vencedor = resultados.iloc[0]
params = json.loads(vencedor["params"])
mapa_cls = {
    "k-means": KMeans,
    "mini-batch-k-means": MiniBatchKMeans,
    "bisecting-k-means": BisectingKMeans,
}
modelo_final = mapa_cls[vencedor["modelo"]](random_state=SEMENTE, **params)
labels = modelo_final.fit_predict(X)

caminho = config.MODELS / f"model_bonus_hr_{vencedor['modelo'].replace('-', '_')}.joblib"
joblib.dump({"modelo": modelo_final, "params": params, "metricas": vencedor.to_dict()}, caminho)

perfil = (
    features.assign(segmento=labels, Attrition=avaliacao["Attrition"].values)
    .groupby("segmento")
    .agg(
        n=("segmento", "size"),
        pct_attrition=("Attrition", lambda s: (s == "Yes").mean() * 100),
        monthly_income=("MonthlyIncome", "median"),
        age=("Age", "median"),
        years_company=("YearsAtCompany", "median"),
        job_satisfaction=("JobSatisfaction", "mean"),
    )
    .round(2)
)
perfil.to_csv(config.TABLES / "bonus_perfil_vencedor.csv")

from sklearn.metrics import adjusted_rand_score

ari_vs_principal = float(adjusted_rand_score(principal["cluster_kmedoids_gower"], labels))
print("Vencedor:", vencedor["modelo"], params)
print("Silhueta:", round(float(vencedor["silhueta"]), 4))
print("ARI vs K-Medoids/Gower:", round(ari_vs_principal, 4))
print("Salvo em:", caminho)
perfil

### Leitura

- O bônus opera no espaço euclidiano padronizado; o modelo oficial permanece Gower.
- Se o ARI contra o K-Medoids for baixo, confirma a divergência já vista na avaliação.
- Use este notebook para sensibilidade — não para substituir a partição publicada.